# FastText API Usage Guide

This notebook demonstrates how to use the FastText API for text classification.
It covers loading a trained model, making predictions on new text, and using
the utility functions provided in fasttext_utils.py.

In [1]:
# Import libraries and utility functions
import fasttext
from sklearn.datasets import fetch_20newsgroups
from fasttext_utils import (
    clean_text,
    prepare_fasttext_data,
    train_fasttext_model,
    evaluate_model,
    get_predictions
)

print("Libraries imported successfully!")

Libraries imported successfully!


## Part 1: Training a Model from Scratch

The FastText API allows you to train a supervised classification model
with a single function call. Here we demonstrate the minimal setup required.

In [3]:
# Load dataset and prepare training data
train_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

prepare_fasttext_data(train_data, '/tmp/train.txt')
prepare_fasttext_data(test_data, '/tmp/test.txt')

# Train a model using the FastText API
model, train_time = train_fasttext_model(
    '/tmp/train.txt',
    epoch=25,
    lr=0.5,
    word_ngrams=2
)
print(f"Model trained in {train_time:.2f} seconds")

Saved 11314 samples to /tmp/train.txt
Saved 7532 samples to /tmp/test.txt
Model trained in 8.88 seconds


## Part 2: Making Predictions on New Text

Once trained, the FastText API allows you to classify any new text
with a single call to model.predict().

In [4]:
# Predict the category of new text samples
sample_texts = [
    "The new graphics card delivers exceptional performance for gaming.",
    "The senator proposed a new bill to regulate firearm purchases.",
    "Scientists discovered a new exoplanet in the habitable zone.",
    "The team won the championship after an outstanding season.",
    "The stock market reached a new all-time high today."
]

print("Predictions on new text samples:")
print("-" * 60)
for text in sample_texts:
    clean = clean_text(text)
    labels, probs = model.predict(clean)
    category = labels[0].replace('__label__', '').replace('_', '.')
    confidence = probs[0]
    print(f"Text: {text[:50]}...")
    print(f"Predicted: {category} (confidence: {confidence:.4f})")
    print()

Predictions on new text samples:
------------------------------------------------------------
Text: The new graphics card delivers exceptional perform...
Predicted: comp.graphics (confidence: 0.9988)

Text: The senator proposed a new bill to regulate firear...
Predicted: talk.politics.misc (confidence: 0.7130)

Text: Scientists discovered a new exoplanet in the habit...
Predicted: rec.autos (confidence: 0.8161)

Text: The team won the championship after an outstanding...
Predicted: rec.sport.hockey (confidence: 0.9982)

Text: The stock market reached a new all-time high today...
Predicted: rec.autos (confidence: 0.5739)



## Part 3: Predicting Multiple Labels

FastText can return the top-k most likely categories for a given text,
which is useful when the correct category is ambiguous.

In [5]:
# Get top-3 predictions for ambiguous texts
ambiguous_texts = [
    "I am looking to buy a used computer with a good graphics card.",
    "The church discussed the role of religion in modern politics.",
    "The new motorcycle engine has impressive technical specifications."
]

print("Top-3 predictions for ambiguous texts:")
print("-" * 60)
for text in ambiguous_texts:
    clean = clean_text(text)
    labels, probs = model.predict(clean, k=3)
    print(f"Text: {text[:55]}...")
    for label, prob in zip(labels, probs):
        category = label.replace('__label__', '').replace('_', '.')
        print(f"  {category}: {prob:.4f}")
    print()

Top-3 predictions for ambiguous texts:
------------------------------------------------------------
Text: I am looking to buy a used computer with a good graphic...
  comp.graphics: 0.7067
  misc.forsale: 0.1715
  sci.electronics: 0.0652

Text: The church discussed the role of religion in modern pol...
  soc.religion.christian: 0.9997
  talk.religion.misc: 0.0002
  alt.atheism: 0.0001

Text: The new motorcycle engine has impressive technical spec...
  rec.autos: 0.8226
  rec.motorcycles: 0.1420
  comp.sys.mac.hardware: 0.0284



## Part 4: Evaluating Model Performance

The FastText API provides a built-in test function to evaluate
model performance on a labeled dataset.

In [6]:
# Evaluate model performance using the FastText API
results = evaluate_model(model, '/tmp/test.txt')

print("Model Evaluation Results:")
print("-" * 40)
print(f"Test samples:  {results['samples']}")
print(f"Precision:     {results['precision']}")
print(f"Recall:        {results['recall']}")
print(f"F1-Score:      {results['f1']}")

Model Evaluation Results:
----------------------------------------
Test samples:  7532
Precision:     0.6022
Recall:        0.6022
F1-Score:      0.6022


## Part 5: Saving and Loading a Model

FastText models can be saved to disk and reloaded for future use
without retraining, which is useful for production deployment.

In [8]:
# Save the model to disk
model.save_model('/tmp/fasttext_api_model.bin')
print("Model saved to /tmp/fasttext_api_model.bin")

# Load the model back
loaded_model = fasttext.load_model('/tmp/fasttext_api_model.bin')
print("Model loaded successfully!")

# Verify the loaded model works correctly
test_text = "The astronauts completed a successful spacewalk outside the station."
clean = clean_text(test_text)
labels, probs = loaded_model.predict(clean)
category = labels[0].replace('__label__', '').replace('_', '.')
print(f"\nTest prediction with loaded model:")
print(f"Text: {test_text}")
print(f"Predicted: {category} (confidence: {probs[0]:.4f})")

Model saved to /tmp/fasttext_api_model.bin
Model loaded successfully!

Test prediction with loaded model:
Text: The astronauts completed a successful spacewalk outside the station.
Predicted: sci.space (confidence: 0.4991)


## Part 6: Word Vectors

FastText also provides access to word vectors, which capture semantic
relationships between words. These can be used for tasks beyond classification
such as similarity search and clustering.

In [9]:
# Access word vectors from the trained model
words = ['politics', 'religion', 'computer', 'sports', 'science']

print("Word vector dimensions:", model.get_dimension())
print()
print("Sample word vectors (first 5 dimensions):")
print("-" * 50)
for word in words:
    vector = model.get_word_vector(word)
    print(f"{word:12s}: {vector[:5].round(4)}")

print()
# Find nearest neighbors for a word
print("Words most similar to 'computer':")
print("-" * 50)
neighbors = model.get_nearest_neighbors('computer', k=5)
for score, word in neighbors:
    print(f"  {word:20s} similarity: {score:.4f}")

Word vector dimensions: 100

Sample word vectors (first 5 dimensions):
--------------------------------------------------
politics    : [ 0.2386 -0.4697 -0.5411  0.3822 -0.2242]
religion    : [-1.381  -0.6163  1.7671  0.565  -0.9371]
computer    : [ 0.1015  0.5626  0.7517 -0.7056  0.1837]
sports      : [ 0.1007 -0.1269 -0.4817  0.0266 -0.5756]
science     : [ 0.159  -0.0033  0.4031 -0.2401  0.9567]

Words most similar to 'computer':
--------------------------------------------------
  lifespan             similarity: 0.9351
  video                similarity: 0.9311
  card                 similarity: 0.9298
  screen               similarity: 0.9208
  available            similarity: 0.9171


## Summary

This notebook demonstrated the key FastText API capabilities:

1. Training a supervised classification model with train_supervised()
2. Making single and top-k predictions with model.predict()
3. Evaluating model performance with model.test()
4. Saving and loading models with save_model() and load_model()
5. Accessing word vectors with get_word_vector() and get_nearest_neighbors()

For a complete walkthrough of the text classification pipeline including
hyperparameter tuning and model comparison, see fasttext.example.ipynb.